# Membership Inference Probing in Language Models
### Can Iterative Paraphrasing Erase Training Data Signals?

**Author:** Mohammad Arifur Rahman  
**Research connection:** Extends Hajishirzi et al. 2026 —  
*Learning to Detect Language Model Training Data via Active Reconstruction*

---
### ✅ Requirements
- Google Colab Free (T4 GPU) OR Local GPU/CPU
- No API keys needed — 100% free HuggingFace models
- Runtime: ~1-2 hours on free Colab

---
### What this does:
1. Loads **member texts** (WikiText-103 — GPT-2 training data)  
2. Loads **non-member texts** (CNN/DailyMail — NOT in GPT-2 training)  
3. Computes **4 behavioral probe signals** using GPT-2 Medium  
4. Applies **3 rounds of paraphrase perturbation** via Pegasus (free HF model)  
5. Measures AUC degradation — does paraphrasing erase membership signals?

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────
!pip install -q transformers datasets scikit-learn sentencepiece
print('Done.')

In [ ]:
# ── Cell 2: Config — change these if needed ───────────────────────────────

# Probe model — GPT-2 is confirmed trained on WebText/Wikipedia
PROBE_MODEL      = 'gpt2-medium'          # ~345M params, fits free Colab

# Paraphrase model — Pegasus XSUM is free, no API needed
PARAPHRASE_MODEL = 'google/pegasus-xsum'  # strong paraphraser, HF hosted

# Experiment scale
N_TEXTS           = 100   # texts per class — 100 runs in ~1.5hrs on free Colab
                          # use 30 for a quick test run first
PARAPHRASE_ROUNDS = 3
MAX_TOKENS        = 256
OUTPUT_DIR        = 'results'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Config set.')

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
import math, zlib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    PegasusTokenizer, PegasusForConditionalGeneration
)
from datasets import load_dataset
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

In [ ]:
# ── Cell 4: Load Data ─────────────────────────────────────────────────────
"""
MEMBER texts     = WikiText-103
                   GPT-2 was trained on WebText which contains Wikipedia.
                   WikiText-103 is a clean Wikipedia benchmark — strong proxy.

NON-MEMBER texts = CNN/DailyMail news articles
                   Confirmed NOT in GPT-2's WebText training data.
                   Post-2019 articles further ensure non-membership.
"""

def clean(text, min_words=40, max_chars=1000):
    """Basic text cleaning and length filter."""
    text = ' '.join(text.split())  # normalize whitespace
    if len(text.split()) < min_words:
        return None
    return text[:max_chars]

def load_wikitext(n):
    """Load member texts from WikiText-103."""
    print('Loading MEMBER texts (WikiText-103)...')
    ds = load_dataset('wikitext', 'wikitext-103-raw-v1',
                      split='train', streaming=True)
    texts = []
    for ex in ds:
        t = clean(ex['text'])
        if t:
            texts.append(t)
        if len(texts) >= n:
            break
    print(f'  Loaded {len(texts)} member texts.')
    return texts

def load_cnn(n):
    """Load non-member texts from CNN/DailyMail."""
    print('Loading NON-MEMBER texts (CNN/DailyMail)...')
    ds = load_dataset('cnn_dailymail', '3.0.0',
                      split='train', streaming=True)
    texts = []
    for ex in ds:
        t = clean(ex['article'])
        if t:
            texts.append(t)
        if len(texts) >= n:
            break
    print(f'  Loaded {len(texts)} non-member texts.')
    return texts

member_texts    = load_wikitext(N_TEXTS)
nonmember_texts = load_cnn(N_TEXTS)
print(f'\nTotal: {len(member_texts)} member + {len(nonmember_texts)} non-member')

In [ ]:
# ── Cell 5: Load Probe Model (GPT-2 Medium) ───────────────────────────────
print(f'Loading probe model: {PROBE_MODEL}...')
probe_tokenizer = AutoTokenizer.from_pretrained(PROBE_MODEL)
probe_model     = AutoModelForCausalLM.from_pretrained(
    PROBE_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE)
probe_model.eval()
print('Probe model ready.')

In [ ]:
# ── Cell 6: Load Paraphrase Model (Pegasus — FREE) ────────────────────────
print(f'Loading paraphrase model: {PARAPHRASE_MODEL}...')
para_tokenizer = PegasusTokenizer.from_pretrained(PARAPHRASE_MODEL)
para_model     = PegasusForConditionalGeneration.from_pretrained(
    PARAPHRASE_MODEL,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32
).to(DEVICE)
para_model.eval()
print('Paraphrase model ready.')

In [ ]:
# ── Cell 7: Behavioral Probe Functions ───────────────────────────────────
"""
4 membership signals — each captures a different aspect of memorization:

1. PERPLEXITY        — model surprise; members = lower PPL
2. LOSS CURVATURE    — std of per-token losses; members = smoother
3. ZLIB RATIO        — model loss / compression ratio (Carlini et al. 2021)
4. LOWERCASE RATIO   — loss(original) / loss(lowercased); members memorized casing
"""

@torch.no_grad()
def get_loss(text):
    """Average cross-entropy loss over tokens."""
    enc = probe_tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=MAX_TOKENS
    ).to(DEVICE)
    labels = enc['input_ids'].clone()
    out    = probe_model(**enc, labels=labels)
    return out.loss.item()

@torch.no_grad()
def get_token_losses(text):
    """Per-token losses for curvature calculation."""
    enc = probe_tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=MAX_TOKENS
    ).to(DEVICE)
    labels = enc['input_ids'].clone()
    out    = probe_model(**enc, labels=labels)

    sl = out.logits[..., :-1, :].contiguous()
    tl = labels[..., 1:].contiguous()
    fn = torch.nn.CrossEntropyLoss(reduction='none')
    return fn(sl.view(-1, sl.size(-1)), tl.view(-1)).cpu().tolist()

def compute_probes(text):
    """Compute all 4 behavioral signals."""
    loss      = get_loss(text)
    ppl       = math.exp(min(loss, 20))           # cap to avoid overflow
    curv      = float(np.std(get_token_losses(text)))
    zlib_b    = len(zlib.compress(text.encode('utf-8')))
    zlib_r    = loss / (zlib_b / max(len(text.encode('utf-8')), 1) + 1e-8)
    lc_loss   = get_loss(text.lower())
    lc_ratio  = loss / (lc_loss + 1e-8)
    return {
        'perplexity':      ppl,
        'loss_curvature':  curv,
        'zlib_ratio':      zlib_r,
        'lowercase_ratio': lc_ratio,
        'raw_loss':        loss
    }

# Quick sanity check
test = compute_probes('The quick brown fox jumps over the lazy dog.')
print('Probe sanity check:', {k: round(v,3) for k,v in test.items()})

In [ ]:
# ── Cell 8: Paraphrase Pipeline (Pegasus — FREE) ──────────────────────────
"""
Uses google/pegasus-xsum for paraphrasing — completely free, no API key.
Pegasus is an abstractive summarization model that also paraphrases well
when prompted with the full text as input.

Increasing beam search diversity across rounds simulates
increasing semantic drift (your existing framework logic).
"""

def paraphrase_pegasus(text, round_num):
    """Paraphrase using Pegasus. Higher round = more diverse output."""
    # Truncate input to Pegasus max
    inputs = para_tokenizer(
        text[:600], return_tensors='pt',
        truncation=True, max_length=512
    ).to(DEVICE)

    # Increase diversity per round via num_beam_groups and diversity_penalty
    diversity = [0.5, 1.5, 3.0][min(round_num, 2)]

    with torch.no_grad():
        out = para_model.generate(
            **inputs,
            max_new_tokens=200,
            num_beams=6,
            num_beam_groups=3,
            diversity_penalty=diversity,
            early_stopping=True,
            no_repeat_ngram_size=3
        )
    return para_tokenizer.decode(out[0], skip_special_tokens=True)

def iterative_perturb(text, rounds):
    """Apply paraphrase iteratively. Returns list of versions [r0, r1, r2, r3]."""
    versions = [text]
    cur = text
    for r in range(rounds):
        cur = paraphrase_pegasus(cur, round_num=r)
        versions.append(cur)
    return versions

# Sanity check
sample = member_texts[0]
v1 = paraphrase_pegasus(sample, 0)
print('Original :', sample[:120])
print('Paraphrase:', v1[:120])

In [ ]:
# ── Cell 9: Run Experiment ────────────────────────────────────────────────
"""
⏱ Runtime estimate (free Colab T4):
  N=30  → ~25 min
  N=100 → ~90 min

Run N=30 first to verify everything works,
then re-run with N=100 for full results.
"""

results = []

for label, texts in [('member', member_texts), ('non_member', nonmember_texts)]:
    print(f'\n── Processing {label.upper()} texts ──')
    for text in tqdm(texts, desc=label):
        try:
            versions = iterative_perturb(text, PARAPHRASE_ROUNDS)
            for r, v in enumerate(versions):
                probes = compute_probes(v)
                results.append({
                    'label': label,
                    'round': r,
                    'text_len': len(v.split()),
                    **probes
                })
        except Exception as e:
            print(f'  Skipped one text: {e}')
            continue

df = pd.DataFrame(results)
df.to_csv(f'{OUTPUT_DIR}/probe_results.csv', index=False)
print(f'\nSaved {len(df)} rows → {OUTPUT_DIR}/probe_results.csv')
df.sample(5)

In [ ]:
# ── Cell 10: Plot 1 — Signal Degradation Curves ───────────────────────────
PROBE_COLS = ['perplexity', 'loss_curvature', 'zlib_ratio', 'lowercase_ratio']
COLORS     = {'member': '#1f4e79', 'non_member': '#c00000'}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(
    'Membership Signal Degradation Under Iterative Paraphrasing\n'
    f'(GPT-2 Medium | {N_TEXTS} member + {N_TEXTS} non-member texts)',
    fontsize=13, fontweight='bold'
)

for ax, probe in zip(axes.flatten(), PROBE_COLS):
    for lbl in ['member', 'non_member']:
        sub   = df[df['label'] == lbl]
        means = sub.groupby('round')[probe].mean()
        stds  = sub.groupby('round')[probe].std()
        color = COLORS[lbl]
        ls    = '-' if lbl == 'member' else '--'
        ax.plot(means.index, means.values,
                label=lbl.replace('_', ' ').title(),
                color=color, ls=ls, lw=2.5, marker='o', markersize=6)
        ax.fill_between(means.index,
                        means - stds, means + stds,
                        alpha=0.12, color=color)
    ax.set_title(probe.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('Paraphrase Round', fontsize=9)
    ax.set_ylabel('Signal Value', fontsize=9)
    ax.set_xticks(range(PARAPHRASE_ROUNDS + 1))
    ax.set_xticklabels([f'R{i}' for i in range(PARAPHRASE_ROUNDS + 1)])
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/signal_degradation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: signal_degradation.png')

In [ ]:
# ── Cell 11: Plot 2 — AUC Degradation (KEY FINDING) ──────────────────────
"""
This is your main result figure.

AUC = how well each probe distinguishes member vs non-member text.
  AUC = 1.0 → perfect membership detection
  AUC = 0.5 → random (signal completely erased)

Research question answered here:
  Does iterative paraphrasing push AUC toward 0.5?
"""

auc_rows = []
for r in range(PARAPHRASE_ROUNDS + 1):
    sub    = df[df['round'] == r].copy()
    y_true = (sub['label'] == 'member').astype(int).values
    for probe in PROBE_COLS:
        # Lower perplexity = more likely member, so negate
        scores = (-sub[probe] if probe == 'perplexity' else sub[probe]).values
        try:
            auc = roc_auc_score(y_true, scores)
        except Exception:
            auc = 0.5
        auc_rows.append({'round': r, 'probe': probe, 'auc': auc})

auc_df = pd.DataFrame(auc_rows)
auc_df.to_csv(f'{OUTPUT_DIR}/auc_results.csv', index=False)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
probe_colors = ['#1f4e79', '#2e75b6', '#c00000', '#375623']
markers      = ['o', 's', '^', 'D']

for (probe, grp), col, mk in zip(auc_df.groupby('probe'), probe_colors, markers):
    ax.plot(grp['round'], grp['auc'],
            label=probe.replace('_', ' ').title(),
            color=col, lw=2.5, marker=mk, markersize=8)

ax.axhline(0.5, color='gray', ls=':', lw=2,
           label='Random baseline (AUC = 0.5)')
ax.fill_between(range(PARAPHRASE_ROUNDS + 1), 0.45, 0.55,
                alpha=0.07, color='gray')

ax.set_title(
    'Membership Inference AUC vs. Paraphrase Round\n'
    'Does paraphrasing erase training data signals? (GPT-2 Medium)',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Paraphrase Round (R0 = original)', fontsize=11)
ax.set_ylabel('AUC — Membership Detection', fontsize=11)
ax.set_xticks(range(PARAPHRASE_ROUNDS + 1))
ax.set_xticklabels([f'R{i}' for i in range(PARAPHRASE_ROUNDS + 1)])
ax.set_ylim(0.4, 1.0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/auc_degradation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: auc_degradation.png')

In [ ]:
# ── Cell 12: Summary Table ────────────────────────────────────────────────
print('═' * 60)
print('RESULTS SUMMARY')
print('═' * 60)

pivot = auc_df.pivot(index='probe', columns='round', values='auc')
pivot.columns = [f'Round {c}' for c in pivot.columns]
pivot['Drop R0→R3'] = pivot['Round 0'] - pivot[f'Round {PARAPHRASE_ROUNDS}']
print('\nAUC per probe per round:')
print(pivot.round(4).to_string())

print('\nMean probe values at Round 0 (original texts):')
print(df[df['round']==0].groupby('label')[PROBE_COLS].mean().round(4).to_string())

print('\n── KEY FINDINGS ──')
for _, row in pivot.iterrows():
    drop = row['Drop R0→R3']
    r0   = row['Round 0']
    flag = '⚠️  Signal ERASED' if row[f'Round {PARAPHRASE_ROUNDS}'] < 0.55 else '✅ Signal ROBUST'
    print(f'  {row.name:20s}: AUC {r0:.3f} → {row[f"Round {PARAPHRASE_ROUNDS}"]:.3f}  (drop={drop:.3f})  {flag}')

## 📋 How to Interpret Your Results

| AUC at R3 | Meaning |
|---|---|
| > 0.70 | Membership signal **survives** paraphrasing — hard to erase |
| 0.55–0.70 | Signal **partially degraded** — paraphrasing helps but doesn't fully erase |
| < 0.55 | Signal **erased** — paraphrasing successfully hides membership |

---

## 📄 How to Frame in Email / Resume

> *"I am investigating whether iterative paraphrase perturbation can erase behavioral membership signals in language models — examining how probe-based detection degrades across perturbation rounds, directly connecting to training data detection challenges raised in your 2026 active reconstruction work."*

---

## 🔬 Possible Extensions (for arXiv paper)
- Repeat with OLMo-1B (Hajishirzi's own model) on Colab Pro
- Compare Pegasus paraphrasing vs. back-translation vs. word substitution
- Test whether the same signals work for detecting AI-generated text (connects to Iyyer)